# Practical 1 - Anomaly detection: Methods and exploration

__Course of Digitalisation for Resource Recovery__

__Prof. dr. Saba Daneshgar__

This practical explores how different anomaly detection methods respond to process data and to their parameter settings. Reusable algorithms and visualisation functions are kept in `anomaly_detection.py` script. This notebook focuses on selecting data, testing different algorithms, interpreting results, and comparing methods.



__Data processing steps__

| Step | Objective |
|---|---|
| Data formatting | Check correct data formats and sampling |
| Data exploration | Inspect statistics, distributions and correlations |
| Data filtering | Detect outliers, anomalies and sensor faults |
| Data filling | Replace missing or invalid data points |


## Setup


In [ ]:
import anomaly_detection as ad

# Folder containing the CSV files used in this practical.
DATA_DIR = "data"


## Loading data


In [ ]:
# Choose one dataset. Uncomment another example if needed.
sensor_data = ad.load_sensor_data("wwtp_synthetic_inf", DATA_DIR, "2017-01-01", "2018-01-01")
# sensor_data = ad.load_sensor_data("wwtp_daily", DATA_DIR, "2016-01-01", "2017-10-06")
# sensor_data = ad.load_sensor_data("wwtp_high_freq", DATA_DIR, "2012-05-05 00:00:05", "2012-05-07 04:47:55")
# sensor_data = ad.load_sensor_data("wwtp_tank_level", DATA_DIR, "2011-06-15 17:08:00", "2011-07-19 06:52:00")

sensor_data.head()


In [ ]:
# Sensor variable to analyze.
variable_name = "QI"

# Data formatting

In [ ]:
# Resampling interval; controls time resolution.
sampling_time = "30min"
sensor_data_resampled = ad.resample_data(sensor_data, sampling_time, method="last")
df_sensor = sensor_data_resampled.dropna()
df_sensor.head()


# Data exploration


In [ ]:
statistics = ad.data_inspection(df_sensor, variable_name)
statistics


# Univariate anomaly detection

For the following methods, change the parameters and observe how the detected anomalies and anomaly scores change. All detectors return a harmonized result table with an `is_anomaly` column.


In [ ]:
sensor_data = df_sensor[[variable_name]].copy()
sensor_data.columns = ["Value"]
sensor_data.head()


## Exercise: implement a physical-range detector yourself

Before using the imported anomaly-detection functions, implement a simple detector. A measurement is anomalous if it falls **strictly below** the lower bound or **strictly above** the upper bound; values exactly on a bound are accepted.

1. Complete `manual_range_detector()` using Boolean comparisons and the `|` (OR) operator. Return a Boolean Series with the same index as the input.
2. Apply it to `sensor_data["Value"]` and count the flagged observations.
3. Complete the plot to show measurements, both limits, and detected anomalies as red crosses.
4. Compare your flags with the imported `ad.physical_range_detector()` in the next cell. Do they agree?

**Think about:** How would changing either bound affect the number of detected anomalies? What happens if the selected limits are inappropriate for the sensor?


In [ ]:
import matplotlib.pyplot as plt

# Lowest physically acceptable measurement.
min_range = 0
# Highest physically acceptable measurement.
max_range = 85000

def manual_range_detector(values, lower, upper):
    """Return True for measurements outside the accepted physical range."""
    anomalies = (values >= upper) | (values <= lower)  # Combine the two comparisons using | (OR).
    return anomalies

values = sensor_data["Value"]
manual_anomalies = manual_range_detector(values, min_range, max_range)

# Count the detected anomalies.
n_anomalies = manual_anomalies.sum()
print(f"Detected anomalies: {n_anomalies}")


In [ ]:
# Plot the measurements, limits and detected anomalies.
fig, ax = plt.subplots(figsize=(11, 4))
ax.set_facecolor("#F3F6FA")
ax.plot(values.index, values, color="#2C5282", linewidth=1, label="Measurements")
ax.axhline(min_range, color="#909090", linestyle="--", label="Lower limit")
ax.axhline(max_range, color="#909090", linestyle="--", label="Upper limit")

# Select and plot only the anomalous observations (red crosses).
ax.scatter(values.index[manual_anomalies], values[manual_anomalies], color="#D62728", marker="x", s=45, label="Detected anomalies", zorder=3)

ax.set_title("Manual physical-range detection")
ax.set_ylabel("Measurement")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout()
plt.show()


## Physical range using the imported detector

Now run the reusable implementation with **the same limits** as your manual detector. Compare the resulting anomaly flags with your own implementation. The remaining methods in this notebook use the imported functions.


In [ ]:
# Lowest physically acceptable measurement.
min_range = 0
# Highest physically acceptable measurement.
max_range = 85000

range_result = ad.physical_range_detector(sensor_data, lower=min_range, upper=max_range)
ad.plot_detector_result(range_result, "Physical range").show()


In [ ]:
# Verify that the manual and imported detectors agree.
print("Identical anomaly flags:", manual_anomalies.equals(range_result["is_anomaly"].reindex(manual_anomalies.index).astype(bool)))

__Questions__
1. Which observations are identified as anomalies? Are they necessarily incorrect measurements, or could they represent genuine extreme operating conditions?
2. How does changing the lower and upper limits affect the detected anomalies? What information would you need to select meaningful limits?

## Shewhart


In [ ]:
# Statistic to monitor: "mean", "median", or "range".
statistic = "mean"

# Number of observations used for local statistics.
window = 4
rolling = True


# Threshold for calculating lower and upper limit, it can be used with rolling or fixed window
threshold = 3.0
limits = "fixed"
limit_window = None



shewhart_result = ad.shewhart_detector(sensor_data, window=window, threshold=threshold, 
                                       statistic=statistic, rolling=rolling, limits=limits, limit_window=limit_window)
ad.plot_detector_result(shewhart_result, "Shewhart test").show()


__Questions__
1. Compare the results obtained using the mean, median and range statistics. What types of changes does each detect, and why might their results differ?
2. Experiment with different subgroup sizes and fixed versus rolling control limits. How do these choices influence sensitivity to sudden changes and gradual drift?

## z-score


In [ ]:
# Number of observations used for local statistics.
window = 200
# Decision cutoff; larger values usually detect fewer anomalies.
threshold = 3

zscore_result = ad.zscore_detector(sensor_data, window=window, threshold=threshold)
ad.plot_detector_result(zscore_result, "Rolling z-score").show()


__Questions__
1. Compare global and rolling Z-score detection. Which anomalies are detected by both methods, and which are detected by only one? Why?
2. Experiment with the window size and Z-score threshold. How do these parameters affect the sensitivity of the detector, particularly around periods of changing process behaviour?

## EWMA


In [ ]:
# EWMA memory; smaller values respond faster to changes.
halflife = 10
# Decision cutoff; larger values usually detect fewer anomalies.
threshold = 2.5

ewma_result = ad.ewma_detector(sensor_data, halflife=halflife, threshold=threshold)
ad.plot_detector_result(ewma_result, "EWMA").show()


__Questions__
1. Which types of changes are highlighted by EWMA? Does it respond differently to isolated spikes and sustained shifts in the measurements?
2. Experiment with different smoothing factors (alpha). How does increasing or decreasing alpha affect the responsiveness of the method and its sensitivity to short-term fluctuations?

## Rate of change


In [ ]:
# Number of time steps used to calculate percentage change.
periods = 1
# Minimum accepted percentage change.
min_roc = -15
# Maximum accepted percentage change.
max_roc = 15

roc_result = ad.rate_of_change_detector(sensor_data, periods=periods, lower=min_roc, upper=max_roc)
ad.plot_detector_result(roc_result, "Rate of change", show_limits=False).show()


__Questions__
1. Which observations are detected by the rate-of-change method but not by the physical-range method? What does this tell you about the difference between their detection principles?
2. How does changing the maximum allowed rate of change influence the results? Could a rapid but legitimate process change be incorrectly identified as anomalous?

## Constant signal

In [ ]:
# Number of points for a flat signal
length = 3

# A threshold for considering the values of the signal to be constant
tolerance = 0.5


const_result = ad.constant_value_detector(sensor_data, length=3, tolerance=0.5)
ad.plot_detector_result(const_result, "Constant signal").show()

__Questions__
1. Examine the periods identified as anomalous by the constant signal detector. Why might a sensor reporting the same value for an extended period indicate a malfunction? Could a constant signal also represent legitimate process behaviour?
2. Experiment with different window sizes and tolerance thresholds for detecting constant values. How do these parameters affect the detection of stuck sensors, particularly when measurements contain small fluctuations or noise?

## IQR


In [ ]:
# Number of observations used for local statistics.
window = 460
# Decision cutoff; larger values usually detect fewer anomalies.
threshold = 1.5

iqr_result = ad.iqr_detector(sensor_data, window=window, threshold=threshold)
ad.plot_detector_result(iqr_result, "Rolling IQR").show()


__Questions__
1. Which observations are identified as anomalies by the IQR method? How do its results compare with those of the Z-score detector?
2. Change the IQR multiplier and the window size. How do these choices affect detection? Why might the IQR method be less influenced by extreme observations than methods based on the mean and standard deviation?

## ARIMA

The model-order search can be computationally expensive. Run it when you want to investigate model selection; otherwise use a previously selected order.


In [ ]:
# Optional model-order search
# min_aic, best_order, order_results = ad.arima_order_search(sensor_data, max_p=5, max_q=5)
# print("Best order:", best_order, "AIC:", min_aic)

# ARIMA (p, d, q) model orders.
order = (0, 0, 1)
# Decision cutoff; larger values usually detect fewer anomalies.
threshold = 3.0
arma_result, arma_model = ad.arima_detector(sensor_data, order=order, threshold=threshold)
ad.plot_detector_result(arma_result, "ARMA residual anomaly detection").show()


__Questions__
1. Examine the observations identified as anomalous based on ARIMA prediction residuals. Are these necessarily extreme measurement values, or can they be values that are simply unexpected given previous observations?
2. Change the ARIMA model orders (p, d, q). How does model complexity affect the predictions, residuals, and detected anomalies? Could an unsuitable model produce misleading anomaly detections?

## Compare the univariate methods


In [ ]:
results = {
    "Range": range_result,
    "Shewhart": shewhart_result,
    "Rate of change": roc_result,
    "Constant signal": const_result,
    "Z-score": zscore_result,
    "IQR": iqr_result,
    "EWMA": ewma_result,
    "ARMA": arma_result,
}

summary_fig, detection_table, overall_anomaly = ad.plot_detection_summary(sensor_data, results)
summary_fig.show()

detection_table.sum().sort_values(ascending=False)


# Multivariate anomaly detection

The current LOF and PCA implementations are kept as reusable helpers for this version of the practical. Their internal implementation can be explored in a later exercise.


## Correlation matrix


In [ ]:
corr = ad.correlation_matrix(df_sensor)
corr


In [ ]:
ad.plot_correlation_matrix(df_sensor).show()


## Local Outlier Factor (LOF)


In [ ]:
# Number of neighboring points defining the LOF local density.
n_neighbors = 100
# Expected anomaly fraction, or auto for automatic LOF cutoff.
contamination = "auto"

lof_result = ad.lof_detector(df_sensor, n_neighbors=n_neighbors, contamination=contamination)
print("Number of detected anomalies:", int(lof_result["is_anomaly"].sum()))


**LOF score diagnostic:** larger LOF scores indicate more unusual local density. Compare the score with the decision limit.


In [ ]:
ad.plot_score_result(lof_result, title="LOF anomaly scores").show()


In [ ]:
# Choose two variables to inspect the multivariate result.
# Variable plotted on the horizontal axis.
# x_variable = df_sensor.columns[0]
# # Variable plotted on the vertical axis.
# y_variable = df_sensor.columns[min(9, len(df_sensor.columns) - 1)]

# ad.plot_multivariate_anomalies(df_sensor, lof_result, x_variable, y_variable).show()


# Select the two variables to visualize (indexing starts at 0).
# Try different combinations to explore the multivariate anomalies.
var_x = 0
var_y = 2

# Plot the selected variables.
ad.plot_multivariate_anomalies(
    df_sensor,
    lof_result,
    x_variable=df_sensor.columns[var_x],
    y_variable=df_sensor.columns[var_y]
).show()

In [ ]:
ad.plot_lof_neighborhood_scores(
    df_sensor,
    lof_result,
    x_variable=df_sensor.columns[var_x],
    y_variable=df_sensor.columns[var_y],
    radius_scale=1000
).show()

In [ ]:
# Select the variable to visualize the time series with anomalies detection using LOF
var_idx = 10
variable = df_sensor.columns[var_idx]

ad.plot_detector_result(
    lof_result,
    title=f"LOF — {variable}",
    values=df_sensor[variable],
    show_score=False,
).show()

__Questions__
1. Examine the detected anomalies in a scatter plot of two selected variables. Are all detected anomalies clearly separated from the other observations? Why might an observation be anomalous in the full multivariate space but appear normal when considering only two variables?
2. Experiment with different values of n_neighbors. How does the neighborhood size influence which observations are detected? What is the difference between using a small neighborhood and a large one?

## PCA and squared prediction error (SPE)


In [ ]:
# Fraction of variance PCA should retain.
explained_variance = 0.90
# Quantile determining the PCA SPE detection limit.
confidence = 0.95

pca_result, pca_info = ad.pca_spe_detector(
    df_sensor,
    explained_variance=explained_variance,
    confidence=confidence,
)

print("Number of components:", pca_info["n_components"])
print("Explained variance:", round(pca_info["explained_variance"], 3))
print("SPE control limit:", round(pca_info["control_limit"], 4))

ad.plot_score_result(
    pca_result,
    title="PCA — squared prediction error (SPE)",
    threshold=pca_info["control_limit"],
).show()


**PCA intermediate diagnostics:** inspect retained variance, the principal-component score space, and the SPE score against its control limit.


In [ ]:
ad.plot_pca_explained_variance(pca_info).show()



In [ ]:
# Select which principal components to visualize.
# PC1 and PC2 explain the largest amounts of variance,
pc_x = 1
pc_y = 2

ad.plot_pca_scores(
    pca_info,
    pca_result,
    pc_x=pc_x,
    pc_y=pc_y
).show()

In [ ]:
# Select the variable to visualize the time series with anomalies detection using PCA
var_idx = 10
variable = df_sensor.columns[var_idx]

ad.plot_detector_result(
    pca_result,
    title=f"PCA — {variable}",
    values=df_sensor[variable],
    show_score=False,
).show()

__Questions__
1. Compare the detected anomalies in the original time series and in the principal component score plots. Are all SPE anomalies clearly visible in the first two principal components? Why might some anomalies not appear unusual in the score plot?
2. Change the number of retained principal components. How does this affect the explained variance, reconstruction error (SPE), and detected anomalies? Why might retaining more components not necessarily improve anomaly detection?